# Bayesian Optimization for Black-Box Functions (Submission 6)

This notebook implements the advanced **"Summit Pursuit"** Bayesian Optimization strategy configured specifically for **Round 6 Queries** across 8 distinct black-box function environments.

### Core Optimization Architecture & Strategy for Round 6:
* **Matern 5/2 Kernel Surface Engine:** Retained to flexibly capture non-linearities and localized landscape irregularities without assuming overly smooth behaviors.
* **Robust Target Scaling:** `normalize_y=True` enables stable convergence over vastly diverse functional target boundaries (ranging from values near $10^{-232}$ to over $4440.0$).
* **High-Rigor Fitting Configuration:** Configured to execute **50 hyperparameter optimization restarts** inside the Gaussian Process Regressor to extract robust length-scale definitions.
* **80 Local Optimization Multi-Starts:** Maximization of the multi-modal Expected Improvement (EI) surface is expanded to **80 random restarts** of the L-BFGS-B local optimizer to ensure fine-grained coordinate precision.
* **"Summit Pursuit" Dynamic Exploration Strategy ($\xi$-Mapping):**
  * **$\xi = 0.3000$ (Aggressive Exploration):** Forces wide-area spatial shifts to escape flat dead-zones or deep negative local basins (Functions 1, 3, 4, and 6).
  * **$\xi = 0.0500$ (Noisy-Likelihood Progress):** Maintains balanced progression over highly stochastic observation regions (Function 2 with adaptive regularizer $\alpha = 10^{-2}$).
  * **$\xi = 0.0001$ (Extreme Exploitation):** Targets and refines established functional peaks (such as the 4440.48 peak in Function 5 and the 2.03 peak in Function 7).
  * **$\xi = 0.00001$ (Ultra-Exploitation Milestone Hunt):** Focuses search intensity directly onto the high-value region of Function 8 to chase the 10.0 target threshold.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enable inline graphing inside Jupyter frameworks
%matplotlib inline 

print("Environment initialized. Round 6 Summit Pursuit optimization engines ready.")

## Bounded Bayesian Optimizer Definition

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Configures regularizer parameters for noise mitigation.
        """
        # Increased alpha for noisy Function 2 (1e-2) to handle stochastic variance, 1e-6 for others
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative histories up to current round and updates the GPR surrogate landscape.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]

        # Matern kernel allows for localized flexibility over real-world irregular landscapes
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        # High rigor (50 restarts) ensures high model parameter fitting accuracy in complex multi-modal spaces
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for a targeted exploration jitter (xi).
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Performs multi-start acquisition optimization across 80 randomized restarts.
        """
        best_x = None
        max_ei = -1
        # 80 restarts ensure we do not miss narrow high-value global peaks
        for _ in range(80):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Multi-Function Execution Loop (Functions 1-8)

We iterate through all active function channels, map their dedicated "Summit Pursuit" exploration variables, fit the high-precision GPR surrogate, and record the proposed point configurations to our local file.

In [ ]:
output_file = "submission_6_results.txt"

# xi_map Logic: "Summit Pursuit" Strategy
# 0.00001: Ultra-Exploitation for Function 8 (Chasing 10.0 milestone)
# 0.0001: Extreme Exploitation for Function 5 (Restoring 4440 peak)
# 0.3: Aggressive Exploration to escape negative/zero regions
xi_map = {
    1: 0.3000, # Still dead zone, need radical jump
    2: 0.0500, # Balanced search for noisy landscape
    3: 0.3000, # Negative, force new region
    4: 0.3000, # Negative, force new region
    5: 0.0001, # EXPLOIT: Returning to the 4440 peak region
    6: 0.3000, # Negative, force new region
    7: 0.0001, # EXPLOIT: Refining the 2.03 peak
    8: 0.00001 # ULTRA-EXPLOIT: Chasing the 10.0 milestone
}

print("Generating Submission 6: The Summit Pursuit...")
print("-" * 55)

notebook_results = {}

with open(output_file, "w") as f:
    for i in range(1, 9):
        func_dir = f"function_{i}"
        
        if not os.path.exists(func_dir):
            print(f"Warning: Target '{func_dir}' not found. Skipping folder context.")
            continue
            
        optimizer = BayesianOptimizer(is_noisy=(i == 2))
        optimizer.load_and_fit(func_dir)
        
        current_xi = xi_map[i]
        next_point = optimizer.propose_next_point(xi=current_xi)
        
        # Formatting to match standard template style: Function i: coordinate-coordinate
        formatted_point = "-".join([f"{val:.6f}" for val in next_point])
        
        result_line = f"Function {i}: {formatted_point}"
        f.write(result_line + "\n")
        
        notebook_results[f"Function {i}"] = (formatted_point, current_xi)
        print(f"{result_line} (xi={current_xi})")

print("-" * 55)
print(f"Submission 6 generated in {output_file}. Ready for the leaderboard!")

### Summary Table of Proposed Queries

In [ ]:
print(f"| {'Target Channel':15} | {'Jitter (xi)':13} | {'Proposed Query Coordinates (Round 6)':55} |")
print(f"| {'-'*15} | {'-'*13} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<13} | {coords:55} |")